# Obstacle Detection Pipeline Test

The pipeline logic now lives in the **`obstacle_detection`** package, so this
notebook and the Gradio app share ONE implementation (no more copy-paste drift).

| Module | Responsibility |
|--------|----------------|
| `config.py` | server paths + all tunable thresholds (`ObstacleConfig`) |
| `models.py` | load + cache SAM / vehicle / DA3 (`get_models`, `reset_models`) |
| `car.py` | identify the car region from SAM2 masks |
| `detector.py` | obstacle-decision logic (`find_obstacle`) + `run_pipeline` |
| `visualize.py` | 5-panel result figure (`render_result_figure`) |

**Logic**
1. Run SAM + vehicle + DA3 on the image.
2. Identify the CAR region from **SAM2 masks** (the vehicle model only seeds *which* SAM blob is the car).
3. Reject background (behind the car) and ground-plane masks via metric depth.
4. Flag a SAM mask as an obstacle only if it is in real **contact** with the car silhouette (not merely sharing its bounding box).

To tune thresholds, edit `obstacle_detection/config.py`, or pass an override per call:
`run_pipeline(path, config=ObstacleConfig(min_car_adjacency=0.08))`.

In [ ]:
import os
import matplotlib.pyplot as plt

from obstacle_detection import (
    run_pipeline,
    render_result_figure,
    ObstacleConfig,
    OBSTACLE_IMAGES_DIR,
    reset_models,   # call reset_models() to force a fresh model load without a kernel restart
)

In [ ]:
os.listdir(OBSTACLE_IMAGES_DIR)

In [ ]:
# Run the pipeline on a test image (change the filename to try others).
test_image_path = os.path.join(OBSTACLE_IMAGES_DIR, "9b8f8464-4921-4ba7-8778-b96fcd174517.png")

# To experiment with thresholds without editing config.py, pass an override, e.g.:
#   result = run_pipeline(test_image_path, config=ObstacleConfig(min_car_adjacency=0.08))
result = run_pipeline(test_image_path)
print(f"Final Pipeline Result: {result['obstacle_exist']}")

# 5-panel view: Original | SAM | Vehicle | Depth | Obstacle Mask
# (same renderer the Gradio app uses).
fig_img = render_result_figure(result)
plt.figure(figsize=(25, 8))
plt.imshow(fig_img)
plt.axis("off")
plt.show()